In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistota typu A 
def unc_A(data):
    '''
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z df do numpy
def df_to_numpy(df, sloupce, od=0, do=None, dtype=float):
    """
    sloupce: jmeno sloupce (str) nebo seznam jmen (list[str])
    od, do:  rozsah radku podle pozice (od vcetne, do bez); do=None znamená až do konce
    """
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    data = df[sloupce].iloc[od:do].to_numpy(dtype=dtype)
    return data[:, 0] if len(sloupce) == 1 else data

# ufloat pro tuple
def tuple_unc_A(*velicina):
    list_velicin = [*velicina]
    return tuple(uf(np.mean(v), unc_A(v)) for v in list_velicin)

# array do latex
def array_to_latex(*array, dig=4, format='f', style='L'):
    for u in array:
        print(f'${u:.{dig}{format}{style}}$')

In [ ]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "1791918306"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_mrizka = pd.read_csv(url)
df_mrizka

In [115]:
# difrakce na mrizce
vln_delka = 632.8*1e-9

df_mrizka_list = ['Vzdálenost mřížka–stínítko xa [mm]', 'y1a vpravo [mm]', 'y1a vlevo [mm]', 'y2a vpravo [mm]', 'y2a vlevo [mm]', 'Vzdálenost mřížka–stínítko xb [mm]', 'y1b vpravo [mm]', 'y1b vlevo [mm]', 'y2b vpravo [mm]', 'y2b vlevo [mm]']
xa, y1a_right, y1a_left, y2a_right, y2a_left, xb, y1b_right, y1b_left, y2b_right, y2b_left = df_to_numpy(df_mrizka, df_mrizka_list, dtype=int).T
y1a = (y1a_left + y1a_right)/2
y2a = (y2a_left + y2a_right)/2
y1b = (y1b_left + y1b_right)/2
y2b = (y2b_left + y2b_right)/2

def difraction_formule(y, x, m):
    sin_alpha = y/((y**2 + x**2)**0.5)
    d = (m * vln_delka)/sin_alpha
    return 1/d/1000

d_a = np.concatenate([difraction_formule(y1a, xa, 1), difraction_formule(y2a, xa, 2)])
d_a = uf(np.mean(d_a), unc_A(d_a))
array_to_latex(d_a, dig=1)
d_b = np.concatenate([difraction_formule(y1b, xb, 1), difraction_formule(y2b, xb, 2)])
d_b = uf(np.mean(d_b), unc_A(d_b))
array_to_latex(d_b, dig=1)

$295.9 \pm 0.2$
$597.1 \pm 0.5$
